In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
data_cleaned = pd.read_csv('D:\VSCODE\Project_YT_Model\youtube_revenue_cleaned.csv')
data_cleaned.head()

,Unnamed: 0,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,ad_revenue_usd,Month,category_Education,...,country_IN,country_UK,country_US,Day_of_Week_Friday,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
0,0,9936,1221,320,26497.214184,2.862137,228086,203.178237,9,False,...,True,False,False,False,False,False,False,False,True,False
1,1,10017,642,346,15209.747445,23.738069,736015,140.880508,9,False,...,False,False,False,False,False,False,True,False,False,False
2,2,10097,1979,187,57332.658498,26.200634,240534,360.134008,11,True,...,False,False,False,False,False,False,False,True,False,False
3,3,10034,1191,242,31334.517771,11.770340,434482,224.638261,1,False,...,False,True,False,False,False,False,False,False,True,False
4,4,9889,1858,477,15665.666434,6.635854,42030,165.514388,4,True,...,False,False,False,False,True,False,False,False,False,False


In [4]:
data_cleaned.drop('Unnamed: 0', axis=1, inplace=True)

In [5]:
# Declaring independent and dependent variables
X = data_cleaned.drop(['ad_revenue_usd'], axis=1)

In [6]:
X.head()

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,Month,category_Education,category_Entertainment,category_Gaming,...,country_IN,country_UK,country_US,Day_of_Week_Friday,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
0,9936,1221,320,26497.214184,2.862137,228086,9,False,True,False,...,True,False,False,False,False,False,False,False,True,False
1,10017,642,346,15209.747445,23.738069,736015,9,False,False,True,...,False,False,False,False,False,False,True,False,False,False
2,10097,1979,187,57332.658498,26.200634,240534,11,True,False,False,...,False,False,False,False,False,False,False,True,False,False
3,10034,1191,242,31334.517771,11.770340,434482,1,False,True,False,...,False,True,False,False,False,False,False,False,True,False
4,9889,1858,477,15665.666434,6.635854,42030,4,True,False,False,...,False,False,False,False,True,False,False,False,False,False


In [7]:
y = data_cleaned['ad_revenue_usd']

y.head()

0    203.178237
1    140.880508
2    360.134008
3    224.638261
4    165.514388
Name: ad_revenue_usd, dtype: float64

In [8]:
#Splitting train and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)

In [9]:
X_train.shape, X_test.shape

((83964, 30), (20991, 30))

In [ ]:
#Scaling the data to have uniform scaling all over the data
scaling = StandardScaler()

X_train_scaled = scaling.fit_transform(X_train)
X_test_scaled = scaling.transform(X_test)


In [10]:
import pickle 

In [11]:
with open("C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_data_scaling.pkl", 'rb') as f:
    scaler = pickle.load(f)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
with open('C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_data_scaling.pkl', 'wb') as f:
    pickle.dump(scaling, f)

Model Training: 
Let's train 5 regression models

1. Linear Regression

In [12]:
from sklearn.linear_model import LinearRegression

In [13]:
li_model = LinearRegression()

In [ ]:
#Training the model
li_model.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
#Testing the model
y_test_pred1 = li_model.predict(X_test_scaled)
y_train_pred1 = li_model.predict(X_train_scaled)

In [17]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [17]:
#Evaluating the model
print(f"""For Test Data:
Mean Absolute Error:{mean_absolute_error(y_test, y_test_pred1)}
Root Mean Squared Error: {root_mean_squared_error(y_test, y_test_pred1)}
r2 score: {r2_score(y_test, y_test_pred1)}
For Trained Data:
Mean Absolute Error:{mean_absolute_error(y_train, y_train_pred1)}
Root Mean Squared Error: {root_mean_squared_error(y_train, y_train_pred1)}
r2 score: {r2_score(y_train, y_train_pred1)}
""")

For Test Data:
Mean Absolute Error:1.8742946840658071e-13
Root Mean Squared Error: 2.1965793556035173e-13
r2 score: 1.0
For Trained Data:
Mean Absolute Error:1.887235491453787e-13
Root Mean Squared Error: 2.2108983018780275e-13
r2 score: 1.0



In [18]:
with open('C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model.pkl', 'wb') as f:
    pickle.dump(li_model, f)

As we are getting r2_score exactly 1.0 which is not normal lets check our model with 'K-Fold Cross-Validation'

In [11]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(li_model, X_train_scaled, y_train, cv=5, scoring='r2')

print('r2 scores per fold:', scores)
print("Average Cross-Validated r2:", scores.mean())

r2 scores per fold: [1. 1. 1. 1. 1.]
Average Cross-Validated r2: 1.0


This says that model is perfectly right

2. Decision Tree

In [26]:
from sklearn.tree import DecisionTreeRegressor

In [27]:
dt_model = DecisionTreeRegressor(random_state=35, max_depth=5)

In [29]:
#Training the model
dt_model.fit(X_train_scaled, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,35
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [30]:
#Testing the model
y_train_pred2 = dt_model.predict(X_train_scaled)
y_test_pred2 = dt_model.predict(X_test_scaled)

In [31]:
#Evaluating the model
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [32]:
print(f"""For Test Data:
Mean Absolute Error:{mean_absolute_error(y_test, y_test_pred2)}
Root Mean Squared Error: {root_mean_squared_error(y_test, y_test_pred2)}
r2 score: {r2_score(y_test, y_test_pred2)}
For Trained Data:
Mean Absolute Error:{mean_absolute_error(y_train, y_train_pred2)}
Root Mean Squared Error: {root_mean_squared_error(y_train, y_train_pred2)}
r2 score: {r2_score(y_train, y_train_pred2)}
""")


For Test Data:
Mean Absolute Error:5.239805658443484
Root Mean Squared Error: 6.4152340270215955
r2 score: 0.9892401483995256
For Trained Data:
Mean Absolute Error:5.154349904638882
Root Mean Squared Error: 6.327146094591632
r2 score: 0.989599544764522



In [33]:
with open('C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model_dt.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

3. Random Forest

In [12]:
from sklearn.ensemble import RandomForestRegressor

In [13]:
RF_model = RandomForestRegressor(n_estimators=10, random_state=42)

In [14]:
#Training the model
RF_model.fit(X_train_scaled, y_train)

,n_estimators,10
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
#Testing the model
y_train_pred3 = RF_model.predict(X_train_scaled)
y_test_pred3 = RF_model.predict(X_test_scaled)

In [18]:
#Evaluating the model
print(f"""For Test Data:
Mean Absolute Error:{mean_absolute_error(y_test, y_test_pred3)}
Root Mean Squared Error: {root_mean_squared_error(y_test, y_test_pred3)}
r2 score: {r2_score(y_test, y_test_pred3)}
For Trained Data:
Mean Absolute Error:{mean_absolute_error(y_train, y_train_pred3)}
Root Mean Squared Error: {root_mean_squared_error(y_train, y_train_pred3)}
r2 score: {r2_score(y_train, y_train_pred3)}
""")

For Test Data:
Mean Absolute Error:0.5782192116395197
Root Mean Squared Error: 0.7390660504295311
r2 score: 0.9998571935643051
For Trained Data:
Mean Absolute Error:0.2648421202894832
Root Mean Squared Error: 0.3526333972959861
r2 score: 0.9999676939763628



In [19]:
joblib.dump(RF_model, 'C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model_RF.pkl', compress=3 )

['C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model_RF.pkl']

In [ ]:
#Let's see what is the contribution of each feature for getting the output
imp_df = pd.DataFrame({
    "Varname": X_train.columns,
    "Imp": RF_model.feature_importances_
})

imp_df.sort_values(by="Imp", ascending=False)

,Varname,Imp
3,watch_time_minutes,0.978036
1,likes,0.020619
2,comments,0.001215
0,views,0.000046
5,subscribers,0.000016
4,video_length_minutes,0.000016
6,Month,0.000011
15,device_TV,0.000002
14,device_Mobile,0.000002
13,device_Desktop,0.000002


4. Gradient Boosting

In [40]:
from sklearn.ensemble import GradientBoostingRegressor

In [41]:
GB_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=45)

In [42]:
#Training the model
GB_model.fit(X_train_scaled, y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [43]:
#Testing the model
y_train_pred4 = GB_model.predict(X_train_scaled)
y_test_pred4 = GB_model.predict(X_test_scaled)

In [44]:
#Evaluating the model
print(f"""For Test Data:
Mean Absolute Error:{mean_absolute_error(y_test, y_test_pred4)}
Root Mean Squared Error: {root_mean_squared_error(y_test, y_test_pred4)}
r2 score: {r2_score(y_test, y_test_pred4)}
For Trained Data:
Mean Absolute Error:{mean_absolute_error(y_train, y_train_pred4)}
Root Mean Squared Error: {root_mean_squared_error(y_train, y_train_pred4)}
r2 score: {r2_score(y_train, y_train_pred4)}
""")

For Test Data:
Mean Absolute Error:0.7088691523569107
Root Mean Squared Error: 0.8904891052931125
r2 score: 0.9997926812916238
For Trained Data:
Mean Absolute Error:0.6852524124853387
Root Mean Squared Error: 0.8604978560813975
r2 score: 0.9998076305030168



In [45]:
with open('C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model_GB.pkl', 'wb') as f:
    pickle.dump(GB_model, f)

5. XGBoost

In [46]:
from xgboost import XGBRegressor

In [47]:
XGB_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state = 42)

In [48]:
#Training the model
XGB_model.fit(X_train_scaled, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [49]:
#Testing the model
y_train_pred5 = XGB_model.predict(X_train_scaled)
y_test_pred5 = XGB_model.predict(X_test_scaled)

In [50]:
#Evaluating the model
print(f"""For Test Data:
Mean Absolute Error:{mean_absolute_error(y_test, y_test_pred5)}
Root Mean Squared Error: {root_mean_squared_error(y_test, y_test_pred5)}
r2 score: {r2_score(y_test, y_test_pred5)}
For Trained Data:
Mean Absolute Error:{mean_absolute_error(y_train, y_train_pred5)}
Root Mean Squared Error: {root_mean_squared_error(y_train, y_train_pred5)}
r2 score: {r2_score(y_train, y_train_pred5)}
""")

For Test Data:
Mean Absolute Error:0.4043307093986776
Root Mean Squared Error: 0.5111196103352805
r2 score: 0.9999316991101888
For Trained Data:
Mean Absolute Error:0.3705878996662417
Root Mean Squared Error: 0.46784649134410017
r2 score: 0.9999431352181095



In [51]:
with open('C:/Users/HP/OneDrive/Desktop/GUVI-DS PROJECT REPORTS/youtube_revenue_model_XGB.pkl', 'wb') as f:
    pickle.dump(XGB_model, f)